# Analize — TRIM-Flux Monte Carlo rezultati

Zbere JSON loge z `results/logs/` (sprotno belezenje iz eval notebookov) in analizira:
- **summarize_by_variant** — mean +/- std AUC po varianti + pacientu (Monte Carlo)
- **pooled ROC** — vse heldout celice v en pool (ozji CI, kot original)
- **reproducibilnost** — ali isti (variant, patient, seed) da isti AUC (seed test)
- **primerjava variant** — je RNA+Flux+TCR nad RNA+TCR?

Variante: `rna_tcr` (original) | `flux_tcr` (Var 1) | `rna_flux_tcr` (Var 2).
Pacienti: izbrani z diagnostiko (>=20 ekspandiranih).

## 0. Mount + importi

In [ ]:
!pip install -q --upgrade numpy scanpy scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LOG_DIR = '/content/drive/MyDrive/Diploma/data/processed/results/logs'
print('Logi:', LOG_DIR, '| obstaja:', os.path.exists(LOG_DIR))

## 1. Agregacijske funkcije

In [ ]:
def collect_logs(log_dir):
    rows = []
    for fn in sorted(os.listdir(log_dir)):
        if not fn.endswith('.json'):
            continue
        try:
            r = json.load(open(os.path.join(log_dir, fn), encoding='utf-8'))
        except Exception:
            continue
        row = {'variant': r.get('variant'), 'patient': r.get('heldout_patient'),
               'seed': r.get('seed'), 'auc': r.get('roc_auc'),
               'n_expanded': r.get('n_expanded_true'), 'timestamp': r.get('timestamp')}
        yo = (r.get('thresholds') or {}).get('youden') or {}
        row['exp_recall'] = yo.get('EXPANDED', {}).get('recall')
        row['exp_precision'] = yo.get('EXPANDED', {}).get('precision')
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_by_variant(log_dir):
    df = collect_logs(log_dir)
    if df.empty:
        print('Ni logov v', log_dir); return df
    g = df.groupby(['variant', 'patient'])['auc'].agg(['mean', 'std', 'count']).reset_index()
    g['mean'] = g['mean'].round(4); g['std'] = g['std'].round(4)
    print(g.to_string(index=False))
    return g


def pooled_roc(log_dir, variant=None):
    from collections import defaultdict
    pools = defaultdict(lambda: {'x': [], 'y': [], 'n': 0}); skipped = 0
    for fn in sorted(os.listdir(log_dir)):
        if not fn.endswith('.json'): continue
        try: r = json.load(open(os.path.join(log_dir, fn), encoding='utf-8'))
        except Exception: continue
        if variant is not None and r.get('variant') != variant: continue
        xt, ys = r.get('x_true'), r.get('y_score')
        if xt is None or ys is None: skipped += 1; continue
        v = r.get('variant'); pools[v]['x'].extend(xt); pools[v]['y'].extend(ys); pools[v]['n'] += 1
    if skipped: print(f'[pooled] {skipped} logov brez x/y preskocenih (stari?).')
    import sklearn.metrics
    out = {}
    for v, d in pools.items():
        x = np.asarray(d['x']); y = np.asarray(d['y']); npos = int((x == 1).sum())
        if not (0 < npos < len(x)):
            print(f'[pooled] {v}: en razred'); continue
        fpr, tpr, _ = sklearn.metrics.roc_curve(x, y, pos_label=1)
        auc = float(sklearn.metrics.auc(fpr, tpr)); out[v] = auc
        print(f'[pooled] {v}: AUC={auc:.4f}  ({d["n"]} zagonov, {len(x)} celic, {npos} ekspand.)')
    return out

## 2. Vsi zagoni (surova tabela)

In [ ]:
df = collect_logs(LOG_DIR)
if df.empty:
    print('Ni logov. Preveri da so eval notebooki tekli in pisali v results/logs/.')
else:
    print(f'Zagonov skupaj: {len(df)}')
    print(df.sort_values(['variant','patient','seed']).to_string(index=False))

## 3. Summarize — mean +/- std AUC (Monte Carlo)

In [ ]:
print('=== mean +/- std AUC po varianti + pacientu ===')
g = summarize_by_variant(LOG_DIR)

## 4. Reproducibilnost (seed test)

In [ ]:
# Ali isti (variant, patient, seed) da IDENTICEN AUC? -> seed dela.
if not df.empty:
    dup = df.groupby(['variant','patient','seed'])['auc'].agg(['count','min','max'])
    dm = dup[dup['count'] > 1]
    if dm.empty:
        print('Ni podvojenih zagonov. Za test: poZeni EN zagon (npr. rna_flux_tcr P24 seed0) 2x.')
        print('Ce sta AUC identicna -> seed dela -> lahko zaupas vsem 45.')
    else:
        for idx, row in dm.iterrows():
            status = 'IDENTICEN (seed dela!)' if row['min']==row['max'] else f'RAZLICEN {row["min"]}!={row["max"]} -> SKRIT vir naklucja!'
            print(f'  {idx}: {int(row["count"])}x -> {status}')

## 5. Pooled ROC (vse celice v en pool)

In [ ]:
print('=== pooled ROC po varianti (cez vse paciente/seede) ===')
pooled = pooled_roc(LOG_DIR)

## 6. Primerjava variant + graf

In [ ]:
if not g.empty:
    # povprecje cez paciente na varianto
    per_variant = g.groupby('variant')['mean'].agg(['mean','std','count'])
    per_variant.columns = ['avg_auc','std_med_pacienti','n_pacientov']
    print('=== PRIMERJAVA VARIANT (povprecje cez paciente) ===')
    print(per_variant.round(4).to_string())
    print()
    print('GLAVNO VPRASANJE: je rna_flux_tcr avg_auc nad rna_tcr? Ce da (+ std ne prekriva) -> flux pomaga.')

    # graf: AUC po varianti, tocke = pacienti
    order = ['rna_tcr','flux_tcr','rna_flux_tcr']
    colors = {'rna_tcr':'#2f6f8f','flux_tcr':'#c56b3e','rna_flux_tcr':'#6b5b95'}
    fig, ax = plt.subplots(figsize=(7,4.5))
    for i, v in enumerate(order):
        sub = g[g['variant']==v]
        if sub.empty: continue
        xs = np.full(len(sub), i) + np.random.uniform(-0.08,0.08,len(sub))
        ax.scatter(xs, sub['mean'], color=colors[v], s=60, alpha=.7, zorder=3, label=f'{v} (n={len(sub)})')
        ax.errorbar(i, sub['mean'].mean(), yerr=sub['mean'].std(), fmt='_', color=colors[v],
                    markersize=30, capsize=6, lw=2, zorder=2)
    ax.axhline(0.5, ls='--', color='gray', lw=1, label='nakljucje')
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=15)
    ax.set_ylabel('ROC AUC (mean po pacientu)'); ax.set_title('AUC po varianti (tocke = pacienti)')
    ax.legend(fontsize=8, loc='lower right'); ax.grid(axis='y', alpha=.3)
    plt.tight_layout(); plt.show()